# Capítulo 3. Análisis exploratorio de datos

**Aprendizaje y Clasificación Automática con R**  
**Autor:** Jesús Gilberto Rodríguez Escobedo

Este cuaderno es **independiente y autónomo**: puede abrirse directamente sin ejecutar capítulos anteriores.

1. Ejecute primero la celda **Preparación automática y autónoma del capítulo**.
2. Después ejecute las celdas en orden.
3. Si Colab reinicia la sesión, vuelva a ejecutar desde la primera celda.

[Volver al índice de cuadernos Colab](https://colab.research.google.com/github/gilbertorodriguez59/libro-machine-learning-r-covid/blob/main/colab/00-indice-colabs.ipynb)


In [ ]:
# Preparación automática y autónoma del capítulo
options(repos = c(CRAN = "https://cloud.r-project.org"))

paquetes_libro <- c(
  "ggplot2", "readr", "dplyr", "tidyr", "stringr", "data.table",
  "class", "rpart", "randomForest", "ranger", "e1071", "naivebayes",
  "neuralnet", "cluster", "caret", "factoextra", "scales", "plotly", "DT"
)
faltantes <- paquetes_libro[!vapply(paquetes_libro, requireNamespace, logical(1), quietly = TRUE)]
if (length(faltantes)) install.packages(faltantes)

dir.create("datos/covid19/procesados", showWarnings = FALSE, recursive = TRUE)
dir.create("datos/covid19/muestras", showWarnings = FALSE, recursive = TRUE)
dir.create("datos/covid19/diccionarios", showWarnings = FALSE, recursive = TRUE)

archivos_colab <- c(
  "util_graficas.R" = "https://raw.githubusercontent.com/gilbertorodriguez59/libro-machine-learning-r-covid/main/util_graficas.R",
  "datos/atus_ml_preparado.csv" = "https://raw.githubusercontent.com/gilbertorodriguez59/libro-machine-learning-r-covid/main/datos/atus_ml_preparado.csv",
  "datos/covid19/procesados/covid19_mexico_2022_ml_preparado.csv.gz" = "https://raw.githubusercontent.com/gilbertorodriguez59/libro-machine-learning-r-covid/main/datos/covid19/procesados/covid19_mexico_2022_ml_preparado.csv.gz",
  "datos/covid19/muestras/covid19_mexico_2022_muestra.csv.gz" = "https://raw.githubusercontent.com/gilbertorodriguez59/libro-machine-learning-r-covid/main/datos/covid19/muestras/covid19_mexico_2022_muestra.csv.gz",
  "datos/covid19/diccionarios/diccionario_covid19_ml.csv" = "https://raw.githubusercontent.com/gilbertorodriguez59/libro-machine-learning-r-covid/main/datos/covid19/diccionarios/diccionario_covid19_ml.csv"
)
for (destino in names(archivos_colab)) {
  if (!file.exists(destino)) download.file(archivos_colab[[destino]], destino, mode = "wb", quiet = TRUE)
}
stopifnot(all(file.exists(names(archivos_colab))))
source("util_graficas.R")
cat("Entorno autónomo listo. R:", R.version.string, "\n")


# Análisis exploratorio de datos

La formulación matemática de **probabilidad, estadística, covarianza y representación multivariada** se desarrolla con mayor profundidad
en los capítulos 2 y 3 de *Fundamentos Matemáticos del Aprendizaje
Automático* [@rodriguez2026fundamentos].

## Objetivos del capítulo

Al finalizar este capítulo, el lector será capaz de cargar una base preparada, revisar su estructura, calcular frecuencias, construir gráficas e interpretar patrones iniciales.

## Cargar paquetes y base


In [ ]:
library(readr)
library(dplyr)
library(ggplot2)
source("util_graficas.R")

ruta_atus_ml <- "datos/atus_ml_preparado.csv"

if (file.exists(ruta_atus_ml)) {
  atus_ml <- read_csv(ruta_atus_ml, show_col_types = FALSE)
  print("Base preparada cargada correctamente.")
} else {
  atus_ml <- NULL
  print("No se encontró el archivo datos/atus_ml_preparado.csv. Ejecuta primero el capítulo 2.")
}


## Explicación del código
Se carga la base preparada en el capítulo anterior. Esto permite empezar directamente con el análisis exploratorio.

## Preparar variables


In [ ]:
if (!is.null(atus_ml)) {
  atus_ml <- atus_ml |>
    mutate(
      ID_HORA_NUM = as.numeric(ID_HORA),
      MES_NUM = as.numeric(MES),
      MES_FACTOR = factor(sprintf("%02d", MES_NUM), levels = sprintf("%02d", 1:12)),
      accidente_con_victimas = factor(accidente_con_victimas, levels = c("Con víctimas", "Solo daños"))
    )
}


## Explicación del código
Se crean variables numéricas y factores ordenados para facilitar tablas y gráficas.

## Distribución de la variable respuesta


In [ ]:
if (!is.null(atus_ml)) {
  tabla_clase <- table(atus_ml$accidente_con_victimas)
  round(100 * prop.table(tabla_clase), 2)
}

if (!is.null(atus_ml)) {
  ggplot(atus_ml, aes(x = accidente_con_victimas, fill = accidente_con_victimas)) +
    geom_bar(width = 0.7) +
    escala_clases_fill() +
    scale_y_continuous(labels = etiqueta_numero) +
    labs(
      title = "Distribución de accidentes según presencia de víctimas",
      subtitle = "Comparación entre clases",
      x = "Clase",
      y = "Número de accidentes",
      fill = "Clase"
    ) +
    tema_libro() +
    theme(legend.position = "none")
}


## Interpretación del resultado
La base está desbalanceada: hay más accidentes de solo daños que accidentes con víctimas. Esto será importante al evaluar modelos.

## Accidentes por mes


In [ ]:
if (!is.null(atus_ml)) {
  accidentes_mes <- atus_ml |> count(MES_FACTOR, name = "n") |> arrange(MES_FACTOR)
  accidentes_mes
}

if (exists("accidentes_mes")) {
  ggplot(accidentes_mes, aes(x = MES_FACTOR, y = n)) +
    geom_col(fill = col_azul, width = 0.75) +
    scale_y_continuous(labels = etiqueta_numero) +
    labs(
      title = "Número de accidentes por mes",
      subtitle = "Base ATUS preparada",
      x = "Mes",
      y = "Número de accidentes"
    ) +
    tema_libro()
}


## Explicación del código
Se cuentan los accidentes por mes y se visualizan con barras. El eje vertical usa separadores de miles para facilitar la lectura.

## Accidentes por hora del día


In [ ]:
if (!is.null(atus_ml)) {
  accidentes_hora <- atus_ml |> count(ID_HORA_NUM, name = "n") |> arrange(ID_HORA_NUM)

  ggplot(accidentes_hora, aes(x = ID_HORA_NUM, y = n)) +
    geom_col(fill = col_turquesa, width = 0.75) +
    scale_x_continuous(breaks = 0:23) +
    scale_y_continuous(labels = etiqueta_numero) +
    labs(
      title = "Número de accidentes por hora del día",
      x = "Hora del día",
      y = "Número de accidentes"
    ) +
    tema_libro()
}


## Interpretación del resultado
La gráfica permite detectar horas con mayor concentración de accidentes.

## Tipos de accidente más frecuentes


In [ ]:
if (!is.null(atus_ml)) {
  accidentes_tipo_top <- atus_ml |> count(TIPACCID, name = "n") |> slice_max(n, n = 10)

  ggplot(accidentes_tipo_top, aes(x = reorder(TIPACCID, n), y = n)) +
    geom_col(fill = col_azul, width = 0.75) +
    coord_flip() +
    scale_y_continuous(labels = etiqueta_numero) +
    labs(
      title = "Tipos de accidente más frecuentes",
      x = "Tipo de accidente",
      y = "Número de accidentes"
    ) +
    tema_libro()
}


## Explicación del código
Se seleccionan los diez tipos de accidente con mayor frecuencia para evitar una gráfica saturada.

## Proporción por tipo de accidente


In [ ]:
if (!is.null(atus_ml)) {
  tipos_top <- atus_ml |> count(TIPACCID) |> slice_max(n, n = 10) |> pull(TIPACCID)
  atus_tipo_top <- atus_ml |> filter(TIPACCID %in% tipos_top)

  ggplot(atus_tipo_top, aes(x = TIPACCID, fill = accidente_con_victimas)) +
    geom_bar(position = "fill") +
    coord_flip() +
    escala_clases_fill() +
    scale_y_continuous(labels = etiqueta_porcentaje) +
    labs(
      title = "Proporción de accidentes con víctimas según tipo de accidente",
      x = "Tipo de accidente",
      y = "Proporción",
      fill = "Clase"
    ) +
    tema_libro()
}


## Interpretación del resultado
Esta gráfica compara proporciones, no cantidades absolutas. Permite identificar tipos de accidente con mayor presencia relativa de víctimas.

## Materiales complementarios del capítulo
Estos recursos permiten repasar los conceptos esenciales del análisis exploratorio de datos mediante distintos formatos.

| Recurso | Utilidad | Abrir o reproducir | Descargar |
|---|---|---|---|
| Video explicativo | Explicación audiovisual del análisis exploratorio de datos y de sus principales herramientas. | [Ver en YouTube](https://www.youtube.com/watch?v=U2hd8iEovCY) | — |
| Presentación en PDF | Diapositivas para lectura, estudio o exposición. | [Ver PDF](recursos/capitulo-03/capitulo-03-analisis-exploratorio-datos.pdf) | [Descargar PDF](recursos/capitulo-03/capitulo-03-analisis-exploratorio-datos.pdf){download="capitulo-03-analisis-exploratorio-datos.pdf"} |
| Presentación editable | Archivo PowerPoint para utilizarlo en clase o adaptarlo. | [Abrir PPTX](recursos/capitulo-03/capitulo-03-analisis-exploratorio-datos.pptx) | [Descargar PPTX](recursos/capitulo-03/capitulo-03-analisis-exploratorio-datos.pptx){download="capitulo-03-analisis-exploratorio-datos.pptx"} |
| Infografía | Síntesis visual de conceptos, procedimientos y gráficos del capítulo. | [Ver infografía](recursos/capitulo-03/capitulo-03-analisis-exploratorio-datos-infografia.png) | [Descargar PNG](recursos/capitulo-03/capitulo-03-analisis-exploratorio-datos-infografia.png){download="capitulo-03-analisis-exploratorio-datos-infografia.png"} |
| Cuaderno Google Colab | Cuaderno autónomo para ejecutar los ejemplos del capítulo sin necesidad de ejecutar los capítulos anteriores. | [Abrir en Google Colab](https://colab.research.google.com/github/gilbertorodriguez59/libro-machine-learning-r-covid/blob/main/colab/03-analisis-exploratorio.ipynb) | — |

### Video explicativo

### Vista previa de la infografía

[![Infografía del capítulo 3](recursos/capitulo-03/capitulo-03-analisis-exploratorio-datos-infografia.png)](recursos/capitulo-03/capitulo-03-analisis-exploratorio-datos-infografia.png)

**Video del capítulo:** <https://www.youtube.com/watch?v=U2hd8iEovCY>

La presentación PDF, el archivo editable y la infografía pueden descargarse desde la versión web del libro.

Este video forma parte de la lista oficial del curso **Aprendizaje y Clasificación Automática con R**.

[Consultar todos los videos del curso](https://www.youtube.com/playlist?list=PLDJYd2v7Kt-Q)

Los materiales complementarios fueron elaborados con apoyo de **NotebookLM de Google**, a partir del contenido del capítulo, y posteriormente revisados y adaptados por el autor. El texto del libro y sus archivos fuente constituyen la referencia principal.

## Laboratorio interactivo: exploración de distribuciones

Selecciona una variable de `iris` y examina su distribución.


**Laboratorio interactivo:** este bloque se ejecuta en la versión web mediante Shinylive; aquí se conserva el desarrollo reproducible del capítulo.


### Laboratorio disponible en la versión web

Permite elegir una variable y visualizar histograma, densidad o boxplot.

## Caso aplicado B: análisis exploratorio de COVID-19

La muestra de 2022 contiene 35 000 registros sin defunción y 15 000 con
defunción registrada. Esta proporción fue construida deliberadamente para
facilitar el aprendizaje de clasificación y **no representa la mortalidad
poblacional real**.


In [ ]:
library(readr)
library(dplyr)
library(ggplot2)

covid <- read_csv(
  "datos/covid19/procesados/covid19_mexico_2022_ml_preparado.csv.gz",
  show_col_types = FALSE
)

covid <- covid |>
  mutate(
    desenlace = factor(
      MURIO,
      levels = c(0, 1),
      labels = c("Sin defunción", "Defunción registrada")
    ),
    paciente = factor(
      TIPO_PACIENTE,
      levels = c(0, 1),
      labels = c("Ambulatorio", "Hospitalizado")
    )
  )


### Distribución de la edad


In [ ]:
ggplot(covid, aes(x = EDAD, fill = desenlace)) +
  geom_histogram(
    bins = 35,
    position = "identity",
    alpha = 0.55
  ) +
  facet_wrap(~ desenlace, ncol = 1) +
  labs(
    x = "Edad",
    y = "Frecuencia",
    fill = "Desenlace"
  ) +
  theme_minimal()


### Mortalidad observada por grupos de edad


In [ ]:
resumen_edad <- covid |>
  mutate(
    grupo_edad = cut(
      EDAD,
      breaks = c(-Inf, 19, 39, 59, 69, 79, Inf),
      labels = c(
        "0-19", "20-39", "40-59",
        "60-69", "70-79", "80 o más"
      )
    )
  ) |>
  group_by(grupo_edad) |>
  summarise(
    registros = n(),
    defunciones = sum(MURIO, na.rm = TRUE),
    proporcion_muestra = mean(MURIO, na.rm = TRUE),
    .groups = "drop"
  )

resumen_edad


### Comorbilidades y desenlace


In [ ]:
covid |>
  group_by(NUM_COMORBILIDADES) |>
  summarise(
    registros = n(),
    proporcion_defuncion_muestra = mean(MURIO, na.rm = TRUE),
    .groups = "drop"
  )


La proporción calculada corresponde a la muestra educativa balanceada. No debe
interpretarse como una tasa epidemiológica de mortalidad.

## Laboratorio interactivo ATUS: mes, hora y víctimas

Este laboratorio utiliza **conteos agregados calculados a partir de la base ATUS preparada del libro**. Permite seleccionar un mes y observar cómo cambia el número de accidentes por hora y la proporción de accidentes con víctimas.


**Laboratorio interactivo:** este bloque se ejecuta en la versión web mediante Shinylive; aquí se conserva el desarrollo reproducible del capítulo.


### Laboratorio interactivo ATUS disponible en la versión web

La versión web permite seleccionar el mes, comparar accidentes por hora y observar el porcentaje de accidentes con víctimas usando agregados de la base ATUS preparada.

## Conclusión

El análisis exploratorio permite detectar patrones iniciales, revisar el desbalance de clases e identificar variables útiles para modelar.
